In [1]:
# Imports
import pandas as pd
import papermill as pm #papermill allows us to execute notebooks not just .py files.
import sys
import os
sys.path.append(os.path.abspath('..')) 
from data_collection import fred_data_download as fred_data_download

In [2]:
#Execute child notebooks
pm.execute_notebook('SP500_notebook.ipynb', 'SP500_notebook_output.ipynb')
pm.execute_notebook('SP500futures_notebook.ipynb', 'SP500futures_notebook_output.ipynb')
fred_data_download.run_fred_download()


Executing:   0%|          | 0/4 [00:00<?, ?cell/s]

Executing:   0%|          | 0/1 [00:00<?, ?cell/s]

            3m_treasury  2yr_treasury  10yr_treasury  yield_curve    VIX  \
date                                                                       
2025-07-30         4.41          3.94           4.38         0.44  15.48   
2025-07-31         4.41          3.94           4.37         0.43  16.72   
2025-08-01         4.35          3.69           4.23         0.54  20.38   
2025-08-02          NaN           NaN            NaN          NaN    NaN   
2025-08-03          NaN           NaN            NaN          NaN    NaN   

            fed_funds_rate  
date                        
2025-07-30            4.33  
2025-07-31            4.33  
2025-08-01            4.33  
2025-08-02            4.33  
2025-08-03            4.33  


In [3]:
#

In [4]:
# Pull DFS
sp = pd.read_parquet('../data/SP500_data.parquet')
fred = pd.read_parquet('../data/fred_data.parquet')
futures = pd.read_csv('../data/Overnight_SP500_data.csv')

In [5]:
fred.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 225 entries, 2025-07-30 to 2026-03-11
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   3m_treasury     153 non-null    float64
 1   2yr_treasury    153 non-null    float64
 2   10yr_treasury   153 non-null    float64
 3   yield_curve     153 non-null    float64
 4   VIX             159 non-null    float64
 5   fed_funds_rate  225 non-null    float64
dtypes: float64(6)
memory usage: 12.3 KB


In [6]:
sp.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 154 entries, 2025-07-30 to 2026-03-10
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Close         154 non-null    float64
 1   High          154 non-null    float64
 2   Low           154 non-null    float64
 3   Open          154 non-null    float64
 4   Volume        154 non-null    int64  
 5   Day Change %  154 non-null    float64
dtypes: float64(5), int64(1)
memory usage: 8.4 KB


In [7]:
fred.index.name = 'Date'

In [8]:
#Perform Left join on date to consolidate dataframes for modeling.
data = pd.merge(sp, fred, on='Date', how='left')

In [9]:
data.head()

,Close,High,Low,Open,Volume,Day Change %,3m_treasury,2yr_treasury,10yr_treasury,yield_curve,VIX,fed_funds_rate
Date,,,,,,,,,,,,
2025-07-30,6362.899902,6396.540039,6336.379883,6381.229980,5375070000,-0.287250,4.41,3.94,4.38,0.44,15.48,4.33
2025-07-31,6339.390137,6427.020020,6327.640137,6427.020020,6077080000,-1.363461,4.41,3.94,4.37,0.43,16.72,4.33
2025-08-01,6238.009766,6287.279785,6212.689941,6287.279785,5827150000,-0.783646,4.35,3.69,4.23,0.54,20.38,4.33
2025-08-04,6329.939941,6330.689941,6271.709961,6271.709961,4842580000,0.928455,4.35,3.69,4.22,0.53,17.52,4.33
2025-08-05,6299.189941,6346.000000,6289.370117,6336.629883,5517410000,-0.590849,4.34,3.72,4.22,0.50,17.85,4.33


In [10]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 154 entries, 2025-07-30 to 2026-03-10
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Close           154 non-null    float64
 1   High            154 non-null    float64
 2   Low             154 non-null    float64
 3   Open            154 non-null    float64
 4   Volume          154 non-null    int64  
 5   Day Change %    154 non-null    float64
 6   3m_treasury     152 non-null    float64
 7   2yr_treasury    152 non-null    float64
 8   10yr_treasury   152 non-null    float64
 9   yield_curve     152 non-null    float64
 10  VIX             154 non-null    float64
 11  fed_funds_rate  154 non-null    float64
dtypes: float64(11), int64(1)
memory usage: 15.6 KB


In [11]:
#show null rows
data[data.isnull().any(axis=1)]
#nulls in 3m_treasury, 2 yr 10 yr, vix ,fed funds, solve with ffill
data = data.fillna(method='ffill')

/var/folders/3n/hh67zc0j3vl1gh8dpln88fv40000gn/T/ipykernel_30927/1399603370.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method='ffill')


In [12]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 154 entries, 2025-07-30 to 2026-03-10
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Close           154 non-null    float64
 1   High            154 non-null    float64
 2   Low             154 non-null    float64
 3   Open            154 non-null    float64
 4   Volume          154 non-null    int64  
 5   Day Change %    154 non-null    float64
 6   3m_treasury     154 non-null    float64
 7   2yr_treasury    154 non-null    float64
 8   10yr_treasury   154 non-null    float64
 9   yield_curve     154 non-null    float64
 10  VIX             154 non-null    float64
 11  fed_funds_rate  154 non-null    float64
dtypes: float64(11), int64(1)
memory usage: 15.6 KB


In [13]:
futures.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 186 entries, 0 to 185
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Date                  186 non-null    object 
 1   Overnight_Return      186 non-null    float64
 2   Overnight_Volatility  186 non-null    float64
 3   Futures_Last_Price    186 non-null    float64
dtypes: float64(3), object(1)
memory usage: 5.9+ KB


In [14]:
#convert futures to date time index
futures['Date'] = pd.to_datetime(futures['Date'])
futures.index = futures['Date']
futures = futures.drop(columns=['Date'])

In [15]:
#Merge with consolidated data
data = pd.merge(data, futures, on='Date', how='left')

In [16]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 154 entries, 2025-07-30 to 2026-03-10
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Close                 154 non-null    float64
 1   High                  154 non-null    float64
 2   Low                   154 non-null    float64
 3   Open                  154 non-null    float64
 4   Volume                154 non-null    int64  
 5   Day Change %          154 non-null    float64
 6   3m_treasury           154 non-null    float64
 7   2yr_treasury          154 non-null    float64
 8   10yr_treasury         154 non-null    float64
 9   yield_curve           154 non-null    float64
 10  VIX                   154 non-null    float64
 11  fed_funds_rate        154 non-null    float64
 12  Overnight_Return      152 non-null    float64
 13  Overnight_Volatility  152 non-null    float64
 14  Futures_Last_Price    152 non-null    float64
dtypes: f

In [17]:
fred.head()

,3m_treasury,2yr_treasury,10yr_treasury,yield_curve,VIX,fed_funds_rate
Date,,,,,,
2025-07-30,4.41,3.94,4.38,0.44,15.48,4.33
2025-07-31,4.41,3.94,4.37,0.43,16.72,4.33
2025-08-01,4.35,3.69,4.23,0.54,20.38,4.33
2025-08-02,NaN,NaN,NaN,NaN,NaN,4.33
2025-08-03,NaN,NaN,NaN,NaN,NaN,4.33


In [18]:
# A few nulls in futures, solve with ffill
data = data.fillna(method='ffill')

/var/folders/3n/hh67zc0j3vl1gh8dpln88fv40000gn/T/ipykernel_30927/899672359.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method='ffill')


In [19]:
data.tail()

,Close,High,Low,Open,Volume,Day Change %,3m_treasury,2yr_treasury,10yr_treasury,yield_curve,VIX,fed_funds_rate,Overnight_Return,Overnight_Volatility,Futures_Last_Price
Date,,,,,,,,,,,,,,,
2026-03-04,6869.500000,6885.939941,6811.640137,6831.689941,5252170000,0.553451,3.71,3.54,4.09,0.55,21.15,3.64,0.008490,0.003031,6825.00
2026-03-05,6830.709961,6870.430176,6770.779785,6851.080078,5989300000,-0.297327,3.70,3.57,4.13,0.56,23.75,3.64,0.006854,0.003002,6871.50
2026-03-06,6740.020020,6773.419922,6711.560059,6769.029785,5793120000,-0.428566,3.69,3.56,4.15,0.59,29.49,3.64,-0.009537,0.003972,6805.50
2026-03-09,6795.990234,6810.439941,6636.040039,6699.799805,6709410000,1.435721,3.71,3.56,4.12,0.56,25.50,3.64,-0.008857,0.005231,6683.25
2026-03-10,6781.479980,6845.080078,6759.740234,6796.560059,5944950000,-0.221878,3.71,3.57,4.15,0.58,24.93,3.64,0.013364,0.004702,6772.50


In [20]:
data.to_csv('../data/consolidated_data.csv', index=False)